In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Content Based
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Collaborative Filtering
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

# Save Model
import pickle

In [ ]:
ratings = pd.read_csv("ratings.csv")
movies = pd.read_csv("movies.csv")

In [ ]:
ratings.head()
movies.head()

In [ ]:
ratings.info()
movies.info()

In [ ]:
ratings.isna().sum()
movies.isna().sum()

In [ ]:
ratings.duplicated().sum()
movies.duplicated().sum()

In [ ]:
ratings = ratings.dropna()
movies = movies.dropna()

In [ ]:
ratings = ratings.drop_duplicates()
movies = movies.drop_duplicates()

In [ ]:
top_movies = ratings['movieId'].value_counts().head(10)

plt.figure(figsize=(10,5))
plt.bar(top_movies.index.astype(str), top_movies.values)
plt.xlabel("Movie ID")
plt.ylabel("Number of Ratings")
plt.title("Top Rated Movies")
plt.show()

In [ ]:
plt.hist(ratings['rating'],
         bins=10,
         edgecolor='black')

plt.xlabel("Ratings")
plt.ylabel("Frequency")
plt.title("Rating Distribution")
plt.show()

In [ ]:
data = pd.merge(ratings, movies, on='movieId')
data.head()

In [ ]:
movies_content = movies[['movieId', 'title', 'genres']]
movies_content.head()

In [ ]:
tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(movies_content['genres'])

print(tfidf_matrix.shape)

In [ ]:
cosine_sim = cosine_similarity(tfidf_matrix)

print(cosine_sim.shape)

In [ ]:
def recommend_movies(movie_title):

    # find movie index
    movie_index = movies_content[movies_content['title'] == movie_title].index[0]

    # similarity scores
    similarity_scores = list(enumerate(cosine_sim[movie_index]))

    # sort movies
    similarity_scores = sorted(similarity_scores,
                               key=lambda x:x[1],
                               reverse=True)

    # top 10 movies
    similarity_scores = similarity_scores[1:11]

    # print recommendations
    for movie in similarity_scores:
        index = movie[0]
        print(movies_content.iloc[index]['title'])

In [ ]:
recommend_movies("Toy Story (1995)")

In [ ]:
reader = Reader(rating_scale=(0.5, 5))

data_surprise = Dataset.load_from_df(
    ratings[['userId', 'movieId', 'rating']],
    reader
)

In [ ]:
trainset, testset = train_test_split(
    data_surprise,
    test_size=0.2,
    random_state=42
)

In [ ]:
model = SVD()

In [ ]:
model.fit(trainset)

In [ ]:
predictions = model.test(testset)

In [ ]:
accuracy.rmse(predictions)

In [ ]:
accuracy.mae(predictions)

In [ ]:
user_id = 1
movie_id = 50

prediction = model.predict(user_id, movie_id)

print("Predicted Rating:", prediction.est)

In [ ]:
def hybrid_recommendation(user_id, movie_title):

    # get movie index
    movie_index = movies_content[movies_content['title'] == movie_title].index[0]

    # cosine similarity
    similarity_scores = list(enumerate(cosine_sim[movie_index]))

    similarity_scores = sorted(similarity_scores,
                               key=lambda x:x[1],
                               reverse=True)

    similarity_scores = similarity_scores[1:21]

    recommendations = []

    for movie in similarity_scores:

        index = movie[0]

        movie_id = movies_content.iloc[index]['movieId']

        title = movies_content.iloc[index]['title']

        # collaborative score
        predicted_rating = model.predict(user_id, movie_id).est

        recommendations.append((title, predicted_rating))

    # sort by predicted rating
    recommendations = sorted(recommendations,
                             key=lambda x:x[1],
                             reverse=True)

    # top 10
    for movie in recommendations[:10]:
        print(movie)

In [ ]:
hybrid_recommendation(1, "Toy Story (1995)")

In [ ]:
pickle.dump(
    movies_content,
    open('movies.pkl', 'wb')
)

In [ ]:
pickle.dump(
    cosine_sim,
    open('cosine.pkl', 'wb')
)

In [ ]:
pickle.dump(
    model,
    open('svd_model.pkl', 'wb')
)

In [ ]:
pip install streamlit

In [ ]:
import streamlit as st
import pickle
import pandas as pd

movies = pickle.load(open('movies.pkl', 'rb'))
cosine_sim = pickle.load(open('cosine.pkl', 'rb'))
model = pickle.load(open('svd_model.pkl', 'rb'))

st.title("Hybrid Movie Recommendation System")

movie_name = st.selectbox(
    "Select Movie",
    movies['title'].values
)

user_id = st.number_input(
    "Enter User ID",
    min_value=1,
    value=1
)

if st.button("Recommend"):

    movie_index = movies[movies['title'] == movie_name].index[0]

    similarity_scores = list(enumerate(cosine_sim[movie_index]))

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x:x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:21]

    recommendations = []

    for movie in similarity_scores:

        index = movie[0]

        movie_id = movies.iloc[index]['movieId']

        title = movies.iloc[index]['title']

        predicted_rating = model.predict(user_id, movie_id).est

        recommendations.append((title, predicted_rating))

    recommendations = sorted(
        recommendations,
        key=lambda x:x[1],
        reverse=True
    )

    st.subheader("Recommended Movies")

    for movie in recommendations[:10]:
        st.write(movie[0])

In [ ]:
streamlit run app.py